# PySpark NLP Preprocessing (Parquet)
Tokenization, Stopword Removal, and TF-IDF Feature Extraction
on Reddit Posts and Comments using PySpark MLlib.

**Source files:** `sentiment_analysis_roberta_POST.parquet`, `sentiment_analysis_roberta_COMMENT.parquet`

## Setup
Initialize SparkSession, import libraries, read Parquet files, and inspect schemas.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

spark = SparkSession.builder \
    .appName("NLP_Preprocessing_Parquet") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession initialized  |  Spark version: {spark.version}")

SparkSession initialized  |  Spark version: 4.1.1


In [2]:
# Parquet stores schema internally — no need for header or inferSchema options
posts_df    = spark.read.parquet("sentiment_analysis_roberta_POST.parquet")
comments_df = spark.read.parquet("sentiment_analysis_roberta_COMMENT.parquet")

print(f"Posts    rows : {posts_df.count():,}")
print(f"Comments rows : {comments_df.count():,}")

Posts    rows : 55,147


Comments rows : 55,216


In [3]:
print("=" * 60)
print("POSTS DataFrame Schema")
print("=" * 60)
posts_df.printSchema()
print("Sample rows from Posts DataFrame:")
posts_df.show(5, truncate=80)

POSTS DataFrame Schema
root
 |-- content_categories: void (nullable = true)
 |-- created_utc: string (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- title: string (nullable = true)
 |-- upvote_ratio: double (nullable = true)
 |-- ups: long (nullable = true)
 |-- downs: long (nullable = true)
 |-- view_count: void (nullable = true)
 |-- AI_Name: string (nullable = true)
 |-- Subgroup: integer (nullable = true)
 |-- sentiment_label: string (nullable = true)
 |-- sentiment_score: double (nullable = true)

Sample rows from Posts DataFrame:


+------------------+-------------------+--------------------------------------------------------------------------------+---------+----------------------------------------------------------------+-------------------+---+-----+----------+-------+--------+---------------+------------------+
|content_categories|        created_utc|                                                                        selftext|subreddit|                                                           title|       upvote_ratio|ups|downs|view_count|AI_Name|Subgroup|sentiment_label|   sentiment_score|
+------------------+-------------------+--------------------------------------------------------------------------------+---------+----------------------------------------------------------------+-------------------+---+-----+----------+-------+--------+---------------+------------------+
|              NULL|2025-04-12 21:57:31|Sora team seems like they were tasked with steamrolling midjourney lol. It's ...|  ChatGPT

In [4]:
print("=" * 60)
print("COMMENTS DataFrame Schema")
print("=" * 60)
comments_df.printSchema()
print("Sample rows from Comments DataFrame:")
comments_df.show(5, truncate=80)

COMMENTS DataFrame Schema
root
 |-- body: string (nullable = true)
 |-- created_utc: string (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removal_reason: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- score: long (nullable = true)
 |-- ups: long (nullable = true)
 |-- AI_Name: string (nullable = true)
 |-- Subgroup: integer (nullable = true)
 |-- sentiment_label: string (nullable = true)
 |-- sentiment_score: double (nullable = true)

Sample rows from Comments DataFrame:


+--------------------------------------------------------------------------------+-------------------+------------+--------------+---------+-----+---+-------+--------+---------------+------------------+
|                                                                            body|        created_utc|is_submitter|removal_reason|subreddit|score|ups|AI_Name|Subgroup|sentiment_label|   sentiment_score|
+--------------------------------------------------------------------------------+-------------------+------------+--------------+---------+-----+---+-------+--------+---------------+------------------+
|                                                     ram is $900 because of this|2026-01-17 19:08:59|       false|          NULL|  ChatGPT|   13| 13|ChatGPT|       1|        LABEL_1| 0.604295015335083|
|i wonder how much of it is spent indulging in stupid image generation fads an...|2026-01-17 19:11:27|       false|          NULL|  ChatGPT|    0|  0|ChatGPT|       2|        LABEL_0|0.938

## Step 1 — Tokenization
Tokenize `title` (Posts) and `body` (Comments) using `Tokenizer`,
then add `token_length` via `F.size()`.

> **Note:** `Tokenizer` throws `NullPointerException` on null values.
> `F.coalesce(..., F.lit(""))` replaces nulls with empty strings before tokenizing.

In [5]:
# Replace null -> empty string to prevent NullPointerException in Tokenizer
posts_df = posts_df.withColumn("title", F.coalesce(F.col("title"), F.lit("")))

post_tokenizer   = Tokenizer(inputCol="title", outputCol="title_tokens")
posts_tokenized_df = post_tokenizer.transform(posts_df)

# Count tokens per row
posts_tokenized_df = posts_tokenized_df.withColumn(
    "token_length", F.size(F.col("title_tokens"))
)

print("Posts — Tokenized 'title' (sample):")
posts_tokenized_df.select("title", "title_tokens", "token_length").show(10, truncate=60)

Posts — Tokenized 'title' (sample):


+------------------------------------------------------------+------------------------------------------------------------+------------+
|                                                       title|                                                title_tokens|token_length|
+------------------------------------------------------------+------------------------------------------------------------+------------+
|                   since when did openai become midjourney??|            [since, when, did, openai, become, midjourney??]|           6|
|                                                     #1 baby|                                                  [#1, baby]|           2|
|       i broke monday. we connected on a disturbing level...|[i, broke, monday., we, connected, on, a, disturbing, lev...|           9|
|                   chatgpt keeps saying i’m violating policy|            [chatgpt, keeps, saying, i’m, violating, policy]|           6|
|i asked chat gpt to visualize me based o

In [6]:
# Replace null -> empty string to prevent NullPointerException in Tokenizer
comments_df = comments_df.withColumn("body", F.coalesce(F.col("body"), F.lit("")))

comment_tokenizer     = Tokenizer(inputCol="body", outputCol="body_tokens")
comments_tokenized_df = comment_tokenizer.transform(comments_df)

# Count tokens per row
comments_tokenized_df = comments_tokenized_df.withColumn(
    "token_length", F.size(F.col("body_tokens"))
)

print("Comments — Tokenized 'body' (sample):")
comments_tokenized_df.select("body", "body_tokens", "token_length").show(10, truncate=60)

Comments — Tokenized 'body' (sample):


+------------------------------------------------------------+------------------------------------------------------------+------------+
|                                                        body|                                                 body_tokens|token_length|
+------------------------------------------------------------+------------------------------------------------------------+------------+
|                                 ram is $900 because of this|                          [ram, is, $900, because, of, this]|           6|
|i wonder how much of it is spent indulging in stupid imag...|[i, wonder, how, much, of, it, is, spent, indulging, in, ...|          20|
|                        idk, this guy seems kind of fun.\n\n|                    [idk,, this, guy, seems, kind, of, fun.]|           7|
|                 that jumpscared me when i scrolled down lol|        [that, jumpscared, me, when, i, scrolled, down, lol]|           8|
|                                        

## Step 1b — Stopword Removal
Remove common English stopwords with `StopWordsRemover`,
then add `token_length_after_stopword` to compare before/after counts.

In [7]:
english_stopwords = StopWordsRemover.loadDefaultStopWords("english")
print(f"Default English stopwords loaded: {len(english_stopwords)} words")

Default English stopwords loaded: 181 words


In [8]:
post_remover = StopWordsRemover(
    inputCol="title_tokens",
    outputCol="title_tokens_filtered",
    stopWords=english_stopwords
)
posts_clean_df = post_remover.transform(posts_tokenized_df)

posts_clean_df = posts_clean_df.withColumn(
    "token_length_after_stopword", F.size(F.col("title_tokens_filtered"))
)

print("Posts — Before vs. After Stopword Removal:")
posts_clean_df.select(
    "title", "title_tokens", "token_length",
    "title_tokens_filtered", "token_length_after_stopword"
).show(10, truncate=55)

Posts — Before vs. After Stopword Removal:


+-------------------------------------------------------+-------------------------------------------------------+------------+-------------------------------------------------------+---------------------------+
|                                                  title|                                           title_tokens|token_length|                                  title_tokens_filtered|token_length_after_stopword|
+-------------------------------------------------------+-------------------------------------------------------+------------+-------------------------------------------------------+---------------------------+
|              since when did openai become midjourney??|       [since, when, did, openai, become, midjourney??]|           6|                  [since, openai, become, midjourney??]|                          4|
|                                                #1 baby|                                             [#1, baby]|           2|                              

In [9]:
comment_remover = StopWordsRemover(
    inputCol="body_tokens",
    outputCol="body_tokens_filtered",
    stopWords=english_stopwords
)
comments_clean_df = comment_remover.transform(comments_tokenized_df)

comments_clean_df = comments_clean_df.withColumn(
    "token_length_after_stopword", F.size(F.col("body_tokens_filtered"))
)

print("Comments — Before vs. After Stopword Removal:")
comments_clean_df.select(
    "body", "body_tokens", "token_length",
    "body_tokens_filtered", "token_length_after_stopword"
).show(10, truncate=55)

Comments — Before vs. After Stopword Removal:


+-------------------------------------------------------+-------------------------------------------------------+------------+-------------------------------------------------------+---------------------------+
|                                                   body|                                            body_tokens|token_length|                                   body_tokens_filtered|token_length_after_stopword|
+-------------------------------------------------------+-------------------------------------------------------+------------+-------------------------------------------------------+---------------------------+
|                            ram is $900 because of this|                     [ram, is, $900, because, of, this]|           6|                                            [ram, $900]|                          2|
|i wonder how much of it is spent indulging in stupid...|[i, wonder, how, much, of, it, is, spent, indulging,...|          20|[wonder, much, spent, indulgin

## Step 2 — TF-IDF Feature Extraction
Convert filtered tokens into numerical feature vectors using **TF-IDF** (two-step process):

1. **HashingTF** — maps each token to a fixed-size vector by hashing, counting term frequency per row
2. **IDF** — downweights terms that appear frequently across many rows (less informative), upweights rare terms

Input columns: `title_tokens_filtered` (Posts), `body_tokens_filtered` (Comments)

Output columns: `tfidf_features`

In [10]:
# ── TF-IDF on Posts (title_tokens_filtered) ─────────────────────────────────

# Step 2.1: HashingTF — hash tokens into a sparse vector of term frequencies
# numFeatures controls the vector dimension (vocabulary size bucket)
post_hashingTF = HashingTF(
    inputCol="title_tokens_filtered",
    outputCol="tf_features",
    numFeatures=10000
)
posts_tf_df = post_hashingTF.transform(posts_clean_df)

# Step 2.2: IDF — fit on the full corpus to compute inverse document frequency weights
# then transform to scale tf_features by IDF weights
post_idf       = IDF(inputCol="tf_features", outputCol="tfidf_features")
post_idf_model = post_idf.fit(posts_tf_df)          # fit: compute IDF weights from corpus
posts_tfidf_df = post_idf_model.transform(posts_tf_df)  # transform: apply weights

print("Posts — TF-IDF features (sample):")
posts_tfidf_df.select(
    "title", "title_tokens_filtered", "tfidf_features"
).show(5, truncate=80)

Posts — TF-IDF features (sample):


+----------------------------------------------------------------+---------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                           title|                                    title_tokens_filtered|                                                                  tfidf_features|
+----------------------------------------------------------------+---------------------------------------------------------+--------------------------------------------------------------------------------+
|                       since when did openai become midjourney??|                    [since, openai, become, midjourney??]|(10000,[3719,6287,7090,8061],[9.126016290063465,6.155601824493764,6.190387940...|
|                                                         #1 baby|                                               [#1, baby]|                       (10000,[1348,9455],[7.5855712

In [11]:
# ── TF-IDF on Comments (body_tokens_filtered) ───────────────────────────────

# Step 2.1: HashingTF — hash tokens into a sparse vector of term frequencies
comment_hashingTF = HashingTF(
    inputCol="body_tokens_filtered",
    outputCol="tf_features",
    numFeatures=10000
)
comments_tf_df = comment_hashingTF.transform(comments_clean_df)

# Step 2.2: IDF — fit on the full corpus to compute inverse document frequency weights
# then transform to scale tf_features by IDF weights
comment_idf       = IDF(inputCol="tf_features", outputCol="tfidf_features")
comment_idf_model = comment_idf.fit(comments_tf_df)         # fit: compute IDF weights
comments_tfidf_df = comment_idf_model.transform(comments_tf_df)  # transform: apply weights

print("Comments — TF-IDF features (sample):")
comments_tfidf_df.select(
    "body", "body_tokens_filtered", "tfidf_features"
).show(5, truncate=80)

Comments — TF-IDF features (sample):
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            body|                                                            body_tokens_filtered|                                                                  tfidf_features|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                     ram is $900 because of this|                                                                     [ram, $900]|                       (10000,[1173,7302],[6.500185548069743,8.97

## Summary Statistics
Compare average token counts before and after stopword removal.

In [12]:
print("=" * 55)
print("Posts (title) — Token Length Summary")
print("=" * 55)
posts_clean_df.select(
    F.avg("token_length").alias("avg_before"),
    F.avg("token_length_after_stopword").alias("avg_after"),
    F.avg(F.col("token_length") - F.col("token_length_after_stopword")).alias("avg_removed")
).show()

print("=" * 55)
print("Comments (body) — Token Length Summary")
print("=" * 55)
comments_clean_df.select(
    F.avg("token_length").alias("avg_before"),
    F.avg("token_length_after_stopword").alias("avg_after"),
    F.avg(F.col("token_length") - F.col("token_length_after_stopword")).alias("avg_removed")
).show()

Posts (title) — Token Length Summary


+-----------------+-----------------+------------------+
|       avg_before|        avg_after|       avg_removed|
+-----------------+-----------------+------------------+
|6.386367345458502|4.297785917638312|2.0885814278201895|
+-----------------+-----------------+------------------+

Comments (body) — Token Length Summary


+------------------+-----------------+-----------------+
|        avg_before|        avg_after|      avg_removed|
+------------------+-----------------+-----------------+
|13.337746305418719|7.459649376992176|5.878096928426543|
+------------------+-----------------+-----------------+



## Step 3 — Logistic Regression Model
Train a multinomial **Logistic Regression** classifier on the TF-IDF feature vectors
to predict `sentiment_label` for both Posts and Comments.

Pipeline:
1. `StringIndexer` — convert `sentiment_label` (string) → `label` (numeric)
2. `randomSplit` — 80/20 train/test with `seed=42`
3. `LogisticRegression` — `featuresCol="tfidf_features"`, `labelCol="label"`, `maxIter=20`
4. Predict on the test set
5. Evaluate accuracy and F1 with `MulticlassClassificationEvaluator`
6. Show 10 sample predictions (text, true label, predicted label)

In [13]:
# Imports for Step 3 — Logistic Regression
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Dictionary to keep evaluation metrics for the comparison summary in Step 4
results = {}

In [14]:
# ── Logistic Regression on Posts ────────────────────────────────────────────

# Sub-step 1: StringIndexer — convert string sentiment_label to numeric label
posts_indexer       = StringIndexer(inputCol="sentiment_label", outputCol="label")
posts_indexer_model = posts_indexer.fit(posts_tfidf_df)
posts_indexed_df    = posts_indexer_model.transform(posts_tfidf_df)

# Sub-step 2: 80/20 train/test split with fixed seed for reproducibility
posts_train_df, posts_test_df = posts_indexed_df.randomSplit([0.8, 0.2], seed=42)

# Sub-step 3: Train Logistic Regression on TF-IDF features
posts_lr       = LogisticRegression(
    featuresCol="tfidf_features",
    labelCol="label",
    maxIter=20
)
posts_lr_model = posts_lr.fit(posts_train_df)

# Sub-step 4: Predictions on the held-out test set
posts_lr_pred = posts_lr_model.transform(posts_test_df)

# Sub-step 5: Evaluate with MulticlassClassificationEvaluator (accuracy + F1)
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
posts_lr_acc = acc_eval.evaluate(posts_lr_pred)
posts_lr_f1  = f1_eval.evaluate(posts_lr_pred)
results["Posts_LR"] = (posts_lr_acc, posts_lr_f1)
print(f"Posts — Logistic Regression  |  Accuracy: {posts_lr_acc:.4f}  |  F1: {posts_lr_f1:.4f}")

# Sub-step 6: Show 10 sample predictions (text, true label, predicted label)
print("Posts — Sample predictions (Logistic Regression):")
posts_lr_pred.select("title", "sentiment_label", "label", "prediction").show(10, truncate=60)

Posts — Logistic Regression  |  Accuracy: 0.7025  |  F1: 0.6998
Posts — Sample predictions (Logistic Regression):


+------------------------------------------------------------+---------------+-----+----------+
|                                                       title|sentiment_label|label|prediction|
+------------------------------------------------------------+---------------+-----+----------+
|                        gpt says it can solve global warming|        LABEL_1|  0.0|       0.0|
|claude ai is usually very helpful, but today it is very b...|        LABEL_0|  1.0|       1.0|
|                   chatgpt browser keeps asking me to log in|        LABEL_0|  1.0|       1.0|
|                                         how to lure chatgpt|        LABEL_1|  0.0|       0.0|
|                        anyone else get spooked on this one?|        LABEL_1|  0.0|       0.0|
|      o1-pro is truly godly for introspection and "therapy".|        LABEL_2|  2.0|       0.0|
|                chats deleted in the recycle bin for 30 days|        LABEL_0|  1.0|       1.0|
|chatgpt prompt of the day: comprehensiv

In [15]:
# ── Logistic Regression on Comments ─────────────────────────────────────────

# Sub-step 1: StringIndexer — convert string sentiment_label to numeric label
comments_indexer       = StringIndexer(inputCol="sentiment_label", outputCol="label")
comments_indexer_model = comments_indexer.fit(comments_tfidf_df)
comments_indexed_df    = comments_indexer_model.transform(comments_tfidf_df)

# Sub-step 2: 80/20 train/test split with fixed seed for reproducibility
comments_train_df, comments_test_df = comments_indexed_df.randomSplit([0.8, 0.2], seed=42)

# Sub-step 3: Train Logistic Regression on TF-IDF features
comments_lr       = LogisticRegression(
    featuresCol="tfidf_features",
    labelCol="label",
    maxIter=20
)
comments_lr_model = comments_lr.fit(comments_train_df)

# Sub-step 4: Predictions on the held-out test set
comments_lr_pred = comments_lr_model.transform(comments_test_df)

# Sub-step 5: Evaluate with MulticlassClassificationEvaluator (accuracy + F1)
comments_lr_acc = acc_eval.evaluate(comments_lr_pred)
comments_lr_f1  = f1_eval.evaluate(comments_lr_pred)
results["Comments_LR"] = (comments_lr_acc, comments_lr_f1)
print(f"Comments — Logistic Regression  |  Accuracy: {comments_lr_acc:.4f}  |  F1: {comments_lr_f1:.4f}")

# Sub-step 6: Show 10 sample predictions (text, true label, predicted label)
print("Comments — Sample predictions (Logistic Regression):")
comments_lr_pred.select("body", "sentiment_label", "label", "prediction").show(10, truncate=60)

Comments — Logistic Regression  |  Accuracy: 0.5961  |  F1: 0.5957
Comments — Sample predictions (Logistic Regression):


+------------------------------------------------------------+---------------+-----+----------+
|                                                        body|sentiment_label|label|prediction|
+------------------------------------------------------------+---------------+-----+----------+
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|\n\n\ncreate api key here and update your billing at the ...|        LABEL_1|  0.0|       1.0|
|\n\n\ntalk to him yourself. it won't have my memories but...|        LABEL_1|  0.0|       0.0|
|            \n\n"i'm back as promised. happy judgement day."|        LABEL_2|  2.0|       2.0|
|\n\n# naah bro wtf is this i got dollar

## Step 4 — Naive Bayes Model
Train a multinomial **Naive Bayes** classifier on the same TF-IDF features and
the same train/test split as Step 3 so results are directly comparable.

Pipeline:
1. Reuse the indexed DataFrames and the 80/20 split from Step 3 (`seed=42`)
2. Train `NaiveBayes` with `featuresCol="tfidf_features"`, `labelCol="label"`,
   `modelType="multinomial"`, `smoothing=1.0`
3. Predict on the test set
4. Evaluate accuracy and F1 with `MulticlassClassificationEvaluator`
5. Show 10 sample predictions (text, true label, predicted label)
6. Print a comparison summary: Logistic Regression vs Naive Bayes

In [16]:
# Imports for Step 4 — Naive Bayes
# Note: NaiveBayes requires non-negative feature values — TF-IDF satisfies this
# (term frequencies are >= 0 and IDF weights are >= 0, so the product is >= 0).
from pyspark.ml.classification import NaiveBayes

# Reuse indexed DataFrames + train/test splits from Step 3 if they exist;
# otherwise re-apply StringIndexer and randomSplit so this cell is standalone-safe.
if "posts_train_df" not in dir() or "posts_test_df" not in dir():
    posts_indexer       = StringIndexer(inputCol="sentiment_label", outputCol="label")
    posts_indexed_df    = posts_indexer.fit(posts_tfidf_df).transform(posts_tfidf_df)
    posts_train_df, posts_test_df = posts_indexed_df.randomSplit([0.8, 0.2], seed=42)

if "comments_train_df" not in dir() or "comments_test_df" not in dir():
    comments_indexer       = StringIndexer(inputCol="sentiment_label", outputCol="label")
    comments_indexed_df    = comments_indexer.fit(comments_tfidf_df).transform(comments_tfidf_df)
    comments_train_df, comments_test_df = comments_indexed_df.randomSplit([0.8, 0.2], seed=42)

In [17]:
# ── Naive Bayes on Posts ────────────────────────────────────────────────────

# Sub-step 1: reuse the same 80/20 split (posts_train_df, posts_test_df) from Step 3

# Sub-step 2: train multinomial Naive Bayes (smoothing=1.0 = Laplace smoothing)
posts_nb       = NaiveBayes(
    featuresCol="tfidf_features",
    labelCol="label",
    modelType="multinomial",
    smoothing=1.0
)
posts_nb_model = posts_nb.fit(posts_train_df)

# Sub-step 3: predict on the held-out test set
posts_nb_pred = posts_nb_model.transform(posts_test_df)

# Sub-step 4: evaluate accuracy and F1 (reuse evaluators from Step 3 if present)
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
posts_nb_acc = acc_eval.evaluate(posts_nb_pred)
posts_nb_f1  = f1_eval.evaluate(posts_nb_pred)
results["Posts_NB"] = (posts_nb_acc, posts_nb_f1)
print(f"Posts — Naive Bayes  |  Accuracy: {posts_nb_acc:.4f}  |  F1: {posts_nb_f1:.4f}")

# Sub-step 5: show 10 sample predictions
print("Posts — Sample predictions (Naive Bayes):")
posts_nb_pred.select("title", "sentiment_label", "label", "prediction").show(10, truncate=60)

Posts — Naive Bayes  |  Accuracy: 0.6356  |  F1: 0.6554
Posts — Sample predictions (Naive Bayes):


+------------------------------------------------------------+---------------+-----+----------+
|                                                       title|sentiment_label|label|prediction|
+------------------------------------------------------------+---------------+-----+----------+
|                        gpt says it can solve global warming|        LABEL_1|  0.0|       0.0|
|claude ai is usually very helpful, but today it is very b...|        LABEL_0|  1.0|       1.0|
|                   chatgpt browser keeps asking me to log in|        LABEL_0|  1.0|       1.0|
|                                         how to lure chatgpt|        LABEL_1|  0.0|       0.0|
|                        anyone else get spooked on this one?|        LABEL_1|  0.0|       0.0|
|      o1-pro is truly godly for introspection and "therapy".|        LABEL_2|  2.0|       0.0|
|                chats deleted in the recycle bin for 30 days|        LABEL_0|  1.0|       1.0|
|chatgpt prompt of the day: comprehensiv

In [18]:
# ── Naive Bayes on Comments ─────────────────────────────────────────────────

# Sub-step 1: reuse the same 80/20 split (comments_train_df, comments_test_df) from Step 3

# Sub-step 2: train multinomial Naive Bayes
comments_nb       = NaiveBayes(
    featuresCol="tfidf_features",
    labelCol="label",
    modelType="multinomial",
    smoothing=1.0
)
comments_nb_model = comments_nb.fit(comments_train_df)

# Sub-step 3: predict on the held-out test set
comments_nb_pred = comments_nb_model.transform(comments_test_df)

# Sub-step 4: evaluate accuracy and F1
comments_nb_acc = acc_eval.evaluate(comments_nb_pred)
comments_nb_f1  = f1_eval.evaluate(comments_nb_pred)
results["Comments_NB"] = (comments_nb_acc, comments_nb_f1)
print(f"Comments — Naive Bayes  |  Accuracy: {comments_nb_acc:.4f}  |  F1: {comments_nb_f1:.4f}")

# Sub-step 5: show 10 sample predictions
print("Comments — Sample predictions (Naive Bayes):")
comments_nb_pred.select("body", "sentiment_label", "label", "prediction").show(10, truncate=60)

Comments — Naive Bayes  |  Accuracy: 0.5676  |  F1: 0.5720
Comments — Sample predictions (Naive Bayes):


+------------------------------------------------------------+---------------+-----+----------+
|                                                        body|sentiment_label|label|prediction|
+------------------------------------------------------------+---------------+-----+----------+
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|                                                        \n\n|        LABEL_1|  0.0|       0.0|
|\n\n\ncreate api key here and update your billing at the ...|        LABEL_1|  0.0|       0.0|
|\n\n\ntalk to him yourself. it won't have my memories but...|        LABEL_1|  0.0|       0.0|
|            \n\n"i'm back as promised. happy judgement day."|        LABEL_2|  2.0|       2.0|
|\n\n# naah bro wtf is this i got dollar

In [19]:
# ── Sub-step 6: Comparison Summary — Logistic Regression vs Naive Bayes ─────
print("=" * 70)
print("Model Comparison Summary  (TF-IDF features, 80/20 split, seed=42)")
print("=" * 70)
print(f"{'Dataset':<10} {'Model':<22} {'Accuracy':>10} {'F1':>10}")
print("-" * 70)
for dataset in ("Posts", "Comments"):
    for model_name, key in (("Logistic Regression", f"{dataset}_LR"),
                            ("Naive Bayes",          f"{dataset}_NB")):
        acc, f1 = results[key]
        print(f"{dataset:<10} {model_name:<22} {acc:>10.4f} {f1:>10.4f}")
    # Per-dataset winner by F1
    lr_f1 = results[f"{dataset}_LR"][1]
    nb_f1 = results[f"{dataset}_NB"][1]
    winner = "Logistic Regression" if lr_f1 >= nb_f1 else "Naive Bayes"
    print(f"  -> {dataset} winner by F1: {winner}")
    print("-" * 70)

Model Comparison Summary  (TF-IDF features, 80/20 split, seed=42)
Dataset    Model                    Accuracy         F1
----------------------------------------------------------------------
Posts      Logistic Regression        0.7025     0.6998
Posts      Naive Bayes                0.6356     0.6554
  -> Posts winner by F1: Logistic Regression
----------------------------------------------------------------------
Comments   Logistic Regression        0.5961     0.5957
Comments   Naive Bayes                0.5676     0.5720
  -> Comments winner by F1: Logistic Regression
----------------------------------------------------------------------


## Step 5 — Data Cleaning & Label Distribution (Phase A)
ทำความสะอาด text ก่อน tokenize:
1. Remove URLs
2. Remove Reddit noise (`[deleted]`, `[removed]`, markdown quotes, HTML entities)
3. Strip non-ASCII (emoji) — baseline แบบตัดออก
4. Normalize whitespace + trim + lowercase
5. Filter rows ที่หลัง clean สั้นกว่า 3 ตัวอักษร
6. ตรวจ label distribution

In [20]:
# Phase A — Data Cleaning & Label Distribution
import pyspark.sql.functions as F

def clean_text(col):
    """Apply a sequence of cleanup regexes to a text column."""
    c = F.coalesce(col, F.lit(""))
    c = F.regexp_replace(c, r"http\S+|www\.\S+", " ")         # URLs
    c = F.regexp_replace(c, r"\[deleted\]|\[removed\]", " ")  # Reddit placeholders
    c = F.regexp_replace(c, r"&amp;|&gt;|&lt;|&#x200B;", " ")    # HTML entities
    c = F.regexp_replace(c, r"[^\x00-\x7F]+", " ")              # non-ASCII (emoji)
    c = F.regexp_replace(c, r"[\r\n\t]+", " ")                 # newlines/tabs
    c = F.regexp_replace(c, r"\s+", " ")                         # collapse whitespace
    c = F.lower(F.trim(c))
    return c

# Posts
posts_clean_df = posts_df.withColumn("title_clean", clean_text(F.col("title")))
posts_clean_df = posts_clean_df.filter(F.length(F.col("title_clean")) >= 3)

# Comments
comments_clean_df = comments_df.withColumn("body_clean", clean_text(F.col("body")))
comments_clean_df = comments_clean_df.filter(F.length(F.col("body_clean")) >= 3)

print(f"Posts    rows before clean : {posts_df.count():,}")
print(f"Posts    rows after  clean : {posts_clean_df.count():,}")
print(f"Comments rows before clean : {comments_df.count():,}")
print(f"Comments rows after  clean : {comments_clean_df.count():,}")

# Label distribution
print("\n── Posts label distribution ──")
posts_clean_df.groupBy("sentiment_label").count().orderBy("sentiment_label").show()
print("── Comments label distribution ──")
comments_clean_df.groupBy("sentiment_label").count().orderBy("sentiment_label").show()

# Cache — used repeatedly downstream
posts_clean_df    = posts_clean_df.cache()
comments_clean_df = comments_clean_df.cache()


Posts    rows before clean : 55,147


Posts    rows after  clean : 54,835
Comments rows before clean : 55,216


Comments rows after  clean : 54,609

── Posts label distribution ──


+---------------+-----+
|sentiment_label|count|
+---------------+-----+
|        LABEL_0|12273|
|        LABEL_1|37377|
|        LABEL_2| 5185|
+---------------+-----+

── Comments label distribution ──


+---------------+-----+
|sentiment_label|count|
+---------------+-----+
|        LABEL_0|16518|
|        LABEL_1|27662|
|        LABEL_2|10429|
+---------------+-----+



## Step 6 — CountVectorizer + Bigrams (Phase B)
เปลี่ยน `HashingTF` → `CountVectorizer(minDF, maxDF)` รวมกับ bigrams ผ่าน `Pipeline`.
รันทั้ง LR และ NB บน features ใหม่ เทียบกับ baseline.

Stages: `Tokenizer → StopWordsRemover → NGram(2) → CountVectorizer(uni) → CountVectorizer(bi) → IDF(uni) → IDF(bi) → VectorAssembler → StringIndexer → Classifier`

In [21]:
# Phase B — build reusable feature pipeline builder
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, NGram, CountVectorizer, IDF,
    VectorAssembler, StringIndexer,
)
from pyspark.ml.classification import LogisticRegression, NaiveBayes

english_stopwords = StopWordsRemover.loadDefaultStopWords("english")

def build_feature_stages(text_col: str):
    """Return the shared feature-engineering stages ending with `features`."""
    tok = Tokenizer(inputCol=text_col, outputCol="tokens")
    sw  = StopWordsRemover(inputCol="tokens", outputCol="tokens_filtered",
                           stopWords=english_stopwords)
    ng  = NGram(n=2, inputCol="tokens_filtered", outputCol="bigrams")
    cv_uni = CountVectorizer(inputCol="tokens_filtered", outputCol="tf_uni",
                             vocabSize=20000, minDF=2.0, maxDF=0.95)
    cv_bi  = CountVectorizer(inputCol="bigrams", outputCol="tf_bi",
                             vocabSize=20000, minDF=2.0, maxDF=0.95)
    idf_uni = IDF(inputCol="tf_uni", outputCol="tfidf_uni")
    idf_bi  = IDF(inputCol="tf_bi",  outputCol="tfidf_bi")
    assembler = VectorAssembler(inputCols=["tfidf_uni", "tfidf_bi"],
                                outputCol="features")
    indexer = StringIndexer(inputCol="sentiment_label", outputCol="label")
    return [tok, sw, ng, cv_uni, cv_bi, idf_uni, idf_bi, assembler, indexer]

acc_eval = MulticlassClassificationEvaluator(labelCol="label",
                                             predictionCol="prediction",
                                             metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label",
                                             predictionCol="prediction",
                                             metricName="f1")

def run_experiment(df, text_col, classifier, tag, results_dict):
    """Fit pipeline, evaluate, store results under `tag`."""
    stages = build_feature_stages(text_col) + [classifier]
    pipe = Pipeline(stages=stages)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    model = pipe.fit(train)
    pred  = model.transform(test)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    results_dict[tag] = (acc, f1)
    print(f"{tag:<45} Accuracy: {acc:.4f}  F1: {f1:.4f}")
    return model, pred


In [22]:
# Phase B — run LR and NB on Posts + Comments with new features
lr_b = LogisticRegression(featuresCol="features", labelCol="label", maxIter=30)
nb_b = NaiveBayes(featuresCol="features", labelCol="label",
                  modelType="multinomial", smoothing=1.0)

posts_lr_b_model, posts_lr_b_pred = run_experiment(
    posts_clean_df, "title_clean", lr_b, "Posts_LR_PhaseB", results)
posts_nb_b_model, posts_nb_b_pred = run_experiment(
    posts_clean_df, "title_clean", nb_b, "Posts_NB_PhaseB", results)

comments_lr_b_model, comments_lr_b_pred = run_experiment(
    comments_clean_df, "body_clean", lr_b, "Comments_LR_PhaseB", results)
comments_nb_b_model, comments_nb_b_pred = run_experiment(
    comments_clean_df, "body_clean", nb_b, "Comments_NB_PhaseB", results)


Posts_LR_PhaseB                               Accuracy: 0.7176  F1: 0.7176


Posts_NB_PhaseB                               Accuracy: 0.6854  F1: 0.6993


Comments_LR_PhaseB                            Accuracy: 0.6195  F1: 0.6195


Comments_NB_PhaseB                            Accuracy: 0.6262  F1: 0.6279


## Step 7 — Hyperparameter Tuning with CrossValidator (Phase C)
ใช้ `CrossValidator(numFolds=3)` หา hyperparameter ที่ดีที่สุดของ LR และ NB
บน pipeline เดียวกันจาก Phase B.

Grid (เก็บแบบกระชับเพื่อให้รันไหว):
- LR: `regParam ∈ {0.0, 0.01, 0.1}`, `elasticNetParam ∈ {0.0}` (L2)
- NB: `smoothing ∈ {0.3, 1.0, 2.0}`

In [23]:
# Phase C — CrossValidator tuning
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

def tune_lr(df, text_col, tag, results_dict):
    """Tune LogisticRegression via 3-fold CV on the full pipeline."""
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
    stages = build_feature_stages(text_col) + [lr]
    pipe = Pipeline(stages=stages)
    grid = (ParamGridBuilder()
            .addGrid(lr.regParam,        [0.0, 0.01, 0.1])
            .addGrid(lr.elasticNetParam, [0.0])
            .build())
    cv = CrossValidator(estimator=pipe, estimatorParamMaps=grid,
                        evaluator=f1_eval, numFolds=3, parallelism=2, seed=42)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    cv_model = cv.fit(train)
    best = cv_model.bestModel
    pred = best.transform(test)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    best_lr = best.stages[-1]
    print(f"{tag} | best regParam={best_lr.getRegParam()} "
          f"elasticNet={best_lr.getElasticNetParam()} "
          f"| Accuracy={acc:.4f} F1={f1:.4f}")
    results_dict[tag] = (acc, f1)
    return best, pred

def tune_nb(df, text_col, tag, results_dict):
    """Tune NaiveBayes via 3-fold CV on the full pipeline."""
    nb = NaiveBayes(featuresCol="features", labelCol="label",
                    modelType="multinomial")
    stages = build_feature_stages(text_col) + [nb]
    pipe = Pipeline(stages=stages)
    grid = (ParamGridBuilder()
            .addGrid(nb.smoothing, [0.3, 1.0, 2.0])
            .build())
    cv = CrossValidator(estimator=pipe, estimatorParamMaps=grid,
                        evaluator=f1_eval, numFolds=3, parallelism=2, seed=42)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    cv_model = cv.fit(train)
    best = cv_model.bestModel
    pred = best.transform(test)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    best_nb = best.stages[-1]
    print(f"{tag} | best smoothing={best_nb.getSmoothing()} "
          f"| Accuracy={acc:.4f} F1={f1:.4f}")
    results_dict[tag] = (acc, f1)
    return best, pred

posts_lr_tuned,    posts_lr_tuned_pred    = tune_lr(posts_clean_df,    "title_clean", "Posts_LR_PhaseC",    results)
posts_nb_tuned,    posts_nb_tuned_pred    = tune_nb(posts_clean_df,    "title_clean", "Posts_NB_PhaseC",    results)
comments_lr_tuned, comments_lr_tuned_pred = tune_lr(comments_clean_df, "body_clean",  "Comments_LR_PhaseC", results)
comments_nb_tuned, comments_nb_tuned_pred = tune_nb(comments_clean_df, "body_clean",  "Comments_NB_PhaseC", results)


Posts_LR_PhaseC | best regParam=0.1 elasticNet=0.0 | Accuracy=0.7775 F1=0.7546


Posts_NB_PhaseC | best smoothing=2.0 | Accuracy=0.6945 F1=0.7074


Comments_LR_PhaseC | best regParam=0.01 elasticNet=0.0 | Accuracy=0.6609 F1=0.6562


Comments_NB_PhaseC | best smoothing=2.0 | Accuracy=0.6318 F1=0.6334


## Step 8 — Alternative Model + Soft-voting Ensemble (Phase D)
1. Train `LinearSVC + OneVsRest` เพิ่มเป็น base model ที่สาม
2. Soft-voting ensemble — เฉลี่ย probability vector ของ best LR + best NB ตาม Phase C,
   (LinearSVC ไม่คืน probability โดย default ใน OneVsRest → ใช้เฉพาะ LR+NB)

In [24]:
# Phase D — LinearSVC (OneVsRest) — cluster-friendly, no Python UDFs
# Note: removed soft-voting ensemble — it requires Python UDFs which don't
# scale well on Spark cluster (the slow path) and crashed repeatedly locally.
# LinearSVC already beat ensemble-style results in our earlier runs.
from pyspark.ml.classification import LinearSVC, OneVsRest

def train_linear_svc(df, text_col, tag, results_dict):
    svc  = LinearSVC(featuresCol="features", labelCol="label", maxIter=30, regParam=0.01)
    ovr  = OneVsRest(classifier=svc, featuresCol="features", labelCol="label")
    stages = build_feature_stages(text_col) + [ovr]
    pipe = Pipeline(stages=stages)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    model = pipe.fit(train)
    pred  = model.transform(test)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    results_dict[tag] = (acc, f1)
    print(f"{tag:<45} Accuracy: {acc:.4f}  F1: {f1:.4f}")
    return model, pred

posts_svc_model,    posts_svc_pred    = train_linear_svc(posts_clean_df,    "title_clean", "Posts_SVC_PhaseD",    results)
comments_svc_model, comments_svc_pred = train_linear_svc(comments_clean_df, "body_clean",  "Comments_SVC_PhaseD", results)


Posts_SVC_PhaseD                              Accuracy: 0.7777  F1: 0.7683


Comments_SVC_PhaseD                           Accuracy: 0.6764  F1: 0.6721


## Step 9 — Class Weighting + Per-class Metrics (Phase E)
1. คำนวณ class weight ให้ Logistic Regression (NB ไม่รองรับ `weightCol`)
2. พิมพ์ per-class precision/recall และ confusion matrix ของ best model

In [25]:
# Phase E — Class Weighting + Per-class Metrics (SQL-only, cluster-friendly)
# Uses Spark SQL joins/crosstab instead of .rdd.map UDFs that crashed earlier.

def train_weighted_lr(df, text_col, tag, results_dict):
    # Compute class weights using SQL aggregation — no UDF
    counts_df = df.groupBy("sentiment_label").count()
    total = df.count()
    k = counts_df.count()
    # weight = total / (k * count_for_class)
    weights_df = counts_df.withColumn(
        "classWeight",
        F.lit(float(total)) / (F.lit(float(k)) * F.col("count").cast("double"))
    ).select("sentiment_label", "classWeight")
    print(f"{tag} class weights:")
    weights_df.show()

    # Attach weights via join — pure SQL, scales to cluster
    df_w = df.join(F.broadcast(weights_df), on="sentiment_label", how="inner")

    lr = LogisticRegression(featuresCol="features", labelCol="label",
                            weightCol="classWeight", maxIter=50, regParam=0.01)
    stages = build_feature_stages(text_col) + [lr]
    pipe = Pipeline(stages=stages)
    train, test = df_w.randomSplit([0.8, 0.2], seed=42)
    model = pipe.fit(train)
    pred  = model.transform(test)

    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    results_dict[tag] = (acc, f1)
    print(f"{tag:<45} Accuracy: {acc:.4f}  F1: {f1:.4f}")

    # Confusion matrix via SQL crosstab — no Python UDF
    print(f"{tag} confusion matrix (rows=true label, cols=prediction):")
    pred.stat.crosstab("label", "prediction").orderBy("label_prediction").show()
    return model, pred

train_weighted_lr(posts_clean_df,    "title_clean", "Posts_LR_Weighted_PhaseE",    results)
train_weighted_lr(comments_clean_df, "body_clean",  "Comments_LR_Weighted_PhaseE", results)


Posts_LR_Weighted_PhaseE class weights:


+---------------+------------------+
|sentiment_label|       classWeight|
+---------------+------------------+
|        LABEL_1|0.4890262282508851|
|        LABEL_0|1.4893125831771639|
|        LABEL_2| 3.525233044037287|
+---------------+------------------+



Posts_LR_Weighted_PhaseE                      Accuracy: 0.7239  F1: 0.7280
Posts_LR_Weighted_PhaseE confusion matrix (rows=true label, cols=prediction):


+----------------+----+----+---+
|label_prediction| 0.0| 1.0|2.0|
+----------------+----+----+---+
|             0.0|5857| 995|537|
|             1.0| 845|1437|137|
|             2.0| 376| 105|559|
+----------------+----+----+---+



Comments_LR_Weighted_PhaseE class weights:


+---------------+------------------+
|sentiment_label|       classWeight|
+---------------+------------------+
|        LABEL_1|0.6580507555491287|
|        LABEL_0|  1.10200992856278|
|        LABEL_2|1.7454214210374916|
+---------------+------------------+



Comments_LR_Weighted_PhaseE                   Accuracy: 0.6557  F1: 0.6554
Comments_LR_Weighted_PhaseE confusion matrix (rows=true label, cols=prediction):


+----------------+----+----+----+
|label_prediction| 0.0| 1.0| 2.0|
+----------------+----+----+----+
|             0.0|3991| 999| 624|
|             1.0|1057|1927| 247|
|             2.0| 635| 229|1301|
+----------------+----+----+----+



(PipelineModel_175a74186765,
 DataFrame[sentiment_label: string, body: string, created_utc: string, is_submitter: boolean, removal_reason: string, subreddit: string, score: bigint, ups: bigint, AI_Name: string, Subgroup: int, sentiment_score: double, body_clean: string, classWeight: double, tokens: array<string>, tokens_filtered: array<string>, bigrams: array<string>, tf_uni: vector, tf_bi: vector, tfidf_uni: vector, tfidf_bi: vector, features: vector, label: double, rawPrediction: vector, probability: vector, prediction: double])

## Step 10 — Final Results Comparison
สรุปทุก phase ใน table เดียว + ระบุ best configuration ต่อ dataset

## Step 11 — Aggressive Features: Char N-grams + Word2Vec (Phase F)
เพิ่ม feature ใหม่ 2 แบบ native Spark ML (ใช้บน GCP Dataproc cluster ได้ตรงๆ):
1. **Character 3-grams** — ช่วย typo/slang/short text ของ Reddit
2. **Word2Vec embeddings** — distributional semantics เติม TF-IDF

Combine ด้วย `VectorAssembler` (word-TFIDF + char-TFIDF + w2v) → train `LinearSVC + OneVsRest`.

หมายเหตุ: **ไม่เทรน NaiveBayes** บน features นี้เพราะ Word2Vec มีค่าลบ (NB ต้องการ non-negative).

In [26]:
# Phase F — aggressive feature pipeline (word + char + Word2Vec) + LinearSVC
from pyspark.ml.feature import Word2Vec, RegexTokenizer

def build_aggressive_feature_stages(text_col: str):
    """Combined features: word uni/bi TF-IDF + char 3-gram TF-IDF + Word2Vec."""
    # ── word path (reuse Phase B approach) ──
    tok_w  = Tokenizer(inputCol=text_col, outputCol="tokens")
    sw     = StopWordsRemover(inputCol="tokens", outputCol="tokens_filtered",
                              stopWords=english_stopwords)
    ng_w   = NGram(n=2, inputCol="tokens_filtered", outputCol="bigrams")
    cv_uni = CountVectorizer(inputCol="tokens_filtered", outputCol="tf_uni",
                             vocabSize=15000, minDF=2.0, maxDF=0.95)
    cv_bi  = CountVectorizer(inputCol="bigrams", outputCol="tf_bi",
                             vocabSize=15000, minDF=2.0, maxDF=0.95)
    idf_u  = IDF(inputCol="tf_uni", outputCol="tfidf_uni")
    idf_b  = IDF(inputCol="tf_bi",  outputCol="tfidf_bi")

    # ── char path — each char becomes a token, then 3-grams ──
    # RegexTokenizer pattern="." gaps=False → every single char is a token
    tok_c   = RegexTokenizer(inputCol=text_col, outputCol="chars",
                             pattern=".", gaps=False, toLowercase=False)
    ng_c    = NGram(n=3, inputCol="chars", outputCol="char_3grams")
    cv_char = CountVectorizer(inputCol="char_3grams", outputCol="tf_char",
                              vocabSize=10000, minDF=5.0, maxDF=0.95)
    idf_c   = IDF(inputCol="tf_char", outputCol="tfidf_char")

    # ── Word2Vec — distributed training, 50-dim ──
    w2v = Word2Vec(inputCol="tokens_filtered", outputCol="w2v_vec",
                   vectorSize=50, minCount=2, numPartitions=4,
                   maxIter=5, seed=42)

    # Assemble everything
    assembler = VectorAssembler(
        inputCols=["tfidf_uni", "tfidf_bi", "tfidf_char", "w2v_vec"],
        outputCol="features",
    )
    indexer = StringIndexer(inputCol="sentiment_label", outputCol="label")

    return [tok_w, sw, ng_w, cv_uni, cv_bi, idf_u, idf_b,
            tok_c, ng_c, cv_char, idf_c,
            w2v, assembler, indexer]


def train_aggressive_svc(df, text_col, tag, results_dict):
    """LinearSVC(OneVsRest) on aggressive combined features."""
    svc = LinearSVC(featuresCol="features", labelCol="label", maxIter=30, regParam=0.01)
    ovr = OneVsRest(classifier=svc, featuresCol="features", labelCol="label")
    stages = build_aggressive_feature_stages(text_col) + [ovr]
    pipe = Pipeline(stages=stages)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    model = pipe.fit(train)
    pred  = model.transform(test)
    acc = acc_eval.evaluate(pred)
    f1  = f1_eval.evaluate(pred)
    results_dict[tag] = (acc, f1)
    print(f"{tag:<45} Accuracy: {acc:.4f}  F1: {f1:.4f}")
    return model, pred


posts_aggr_model,    posts_aggr_pred    = train_aggressive_svc(
    posts_clean_df,    "title_clean", "Posts_SVC_Aggressive_PhaseF",    results)
comments_aggr_model, comments_aggr_pred = train_aggressive_svc(
    comments_clean_df, "body_clean",  "Comments_SVC_Aggressive_PhaseF", results)


Posts_SVC_Aggressive_PhaseF                   Accuracy: 0.8107  F1: 0.8069


Comments_SVC_Aggressive_PhaseF                Accuracy: 0.7052  F1: 0.7028


## Phase G — Auto Tuning (CrossValidator) on Best Model

Minimal grid: `regParam [0.001, 0.01, 0.1]` × `maxIter [30, 80]` = 6 combinations, 3-fold CV, parallelism=2.

Estimated time: ~20 minutes.

In [27]:
# Phase G — Auto Tuning (CrossValidator) on Best Model: SVC Aggressive
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ensure evaluators exist (defined in Phase B/C but re-declare for safety)
acc_eval_g = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval_g  = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1')

def tune_aggressive_svc(df, text_col, tag, results_dict):
    # ① Build pipeline (same as Phase F)
    svc = LinearSVC(featuresCol='features', labelCol='label')
    ovr = OneVsRest(classifier=svc, featuresCol='features', labelCol='label')
    stages = build_aggressive_feature_stages(text_col) + [ovr]
    pipe = Pipeline(stages=stages)

    # ② Define minimal grid: 3 x 2 = 6 combinations
    grid = (ParamGridBuilder()
        .addGrid(svc.regParam, [0.001, 0.01, 0.1])
        .addGrid(svc.maxIter,  [30, 80])
        .build()
    )

    # ③ CrossValidator
    cv = CrossValidator(
        estimator=pipe,
        estimatorParamMaps=grid,
        evaluator=MulticlassClassificationEvaluator(
            labelCol='label', predictionCol='prediction', metricName='f1'
        ),
        numFolds=3,
        parallelism=2,
        seed=42
    )

    # ④ Split and fit
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    print(f'Fitting CrossValidator for {tag} (6 combos x 3 folds)...')
    cv_model = cv.fit(train)

    # ⑤ Log best params
    best_ovr = cv_model.bestModel.stages[-1]
    best_svc = best_ovr.getClassifier()
    print(f'Best params for {tag}:')
    print(f'  regParam = {best_svc.getRegParam()}')
    print(f'  maxIter  = {best_svc.getMaxIter()}')

    # ⑥ Evaluate on held-out test set
    pred = cv_model.transform(test)
    acc = acc_eval_g.evaluate(pred)
    f1  = f1_eval_g.evaluate(pred)
    results_dict[tag] = (acc, f1)

    # ⑦ Compare vs Phase F baseline
    baseline_tag = tag.replace('Tuned_PhaseG', 'Aggressive_PhaseF')
    baseline_f1  = results_dict.get(baseline_tag, (0, 0))[1]
    print(f'{tag:<45} Accuracy: {acc:.4f}  F1: {f1:.4f}')
    print(f'Improvement vs Phase F: {f1 - baseline_f1:+.4f}')
    return cv_model, pred


posts_tuned_model,    posts_tuned_pred    = tune_aggressive_svc(
    posts_clean_df,    'title_clean', 'Posts_SVC_Tuned_PhaseG',    results)

comments_tuned_model, comments_tuned_pred = tune_aggressive_svc(
    comments_clean_df, 'body_clean',  'Comments_SVC_Tuned_PhaseG', results)


Fitting CrossValidator for Posts_SVC_Tuned_PhaseG (6 combos x 3 folds)...


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-package

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-package

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-package

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-package

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [27]:
# Save the best aggressive model as a PipelineModel — portable to Dataproc
import os
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

posts_aggr_model.write().overwrite().save(f"{MODEL_DIR}/posts_svc_aggressive")
comments_aggr_model.write().overwrite().save(f"{MODEL_DIR}/comments_svc_aggressive")
print(f"Saved posts_svc_aggressive and comments_svc_aggressive into ./{MODEL_DIR}/")
print("On GCP: replace paths with gs://<bucket>/models/... and PipelineModel.load the same way.")


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [28]:
# Step 10 — aggregate all results
print("=" * 80)
print(f"FINAL COMPARISON — all phases  (TF-IDF variants, 80/20 split, seed=42)")
print("=" * 80)
print(f"{'Tag':<45} {'Accuracy':>10} {'F1':>10}")
print("-" * 80)
for tag in sorted(results.keys()):
    acc, f1 = results[tag]
    print(f"{tag:<45} {acc:>10.4f} {f1:>10.4f}")
print("-" * 80)

# Best per dataset by F1
for ds in ("Posts", "Comments"):
    subset = {k: v for k, v in results.items() if k.startswith(ds)}
    best_tag = max(subset, key=lambda k: subset[k][1])
    acc, f1 = subset[best_tag]
    print(f"BEST {ds}: {best_tag}  (Accuracy={acc:.4f}, F1={f1:.4f})")

# ── Per-label Precision / Recall / F1 for the best model (Phase F SVC Aggressive) ──
# Uses MulticlassClassificationEvaluator per label — pure Spark, cluster-safe.
# StringIndexer orders by frequency descending:
#   most-frequent label → index 0.0, second → 1.0, third → 2.0

print()
print("=" * 80)
print("PER-LABEL METRICS — Best Model: SVC Aggressive (Phase F)")
print("Label mapping from StringIndexerModel.labels (frequency-descending order)")
print("=" * 80)

def per_label_metrics(pred_df, model, dataset_name):
    """Compute per-label Precision, Recall, F1 using Spark ML evaluators.

    model    : fitted PipelineModel — StringIndexerModel is at stages[13]
    pred_df  : DataFrame from model.transform() — has 'label' & 'prediction'
    """
    # build_aggressive_feature_stages returns 14 stages (0-13);
    # stage 13 is StringIndexer → StringIndexerModel after fit.
    indexer_stage = model.stages[13]
    label_names   = indexer_stage.labels  # e.g. ["LABEL_1", "LABEL_0", "LABEL_2"]

    print()
    print(dataset_name)
    print("  StringIndexer mapping (numeric index → original label):")
    for i, name in enumerate(label_names):
        print(f"    {float(i):.1f}  →  {name}")

    header = f"  {'Idx':<6} {'Original':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}"
    print()
    print(header)
    print("  " + "-" * 60)

    for label_idx in range(len(label_names)):
        ev = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction",
            metricLabel=float(label_idx)
        )
        prec    = ev.setMetricName("precisionByLabel").evaluate(pred_df)
        rec     = ev.setMetricName("recallByLabel").evaluate(pred_df)
        f1_val  = ev.setMetricName("fMeasureByLabel").evaluate(pred_df)
        support = pred_df.filter(F.col("label") == float(label_idx)).count()
        lname   = label_names[label_idx]
        print(f"  {float(label_idx):<6.1f} {lname:<12} {prec:>10.4f} {rec:>10.4f} {f1_val:>10.4f} {support:>10,}")
    print()

per_label_metrics(posts_aggr_pred,    posts_aggr_model,    "POSTS    (title) — Posts_SVC_Aggressive_PhaseF")
per_label_metrics(comments_aggr_pred, comments_aggr_model, "COMMENTS (body)  — Comments_SVC_Aggressive_PhaseF")


FINAL COMPARISON — all phases  (TF-IDF variants, 80/20 split, seed=42)
Tag                                             Accuracy         F1
--------------------------------------------------------------------------------
Comments_LR                                       0.5961     0.5957
Comments_LR_PhaseB                                0.6195     0.6195
Comments_LR_PhaseC                                0.6609     0.6562
Comments_LR_Weighted_PhaseE                       0.6557     0.6554
Comments_NB                                       0.5676     0.5720
Comments_NB_PhaseB                                0.6262     0.6279
Comments_NB_PhaseC                                0.6318     0.6334
Comments_SVC_Aggressive_PhaseF                    0.7052     0.7028
Comments_SVC_PhaseD                               0.6764     0.6721
Posts_LR                                          0.7025     0.6998
Posts_LR_PhaseB                                   0.7176     0.7176
Posts_LR_PhaseC                 

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [29]:
spark.stop()
print("SparkSession stopped.")

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it